# HistoOmniST Coverage95 Evidence Package

This notebook audits the generated manuscript-facing evidence package. It is intentionally conservative: formal results, engineering smoke checks, prepared-but-not-run tasks, and unsupported claims remain separated.

In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src" / "histoomnist").exists() and (ROOT.parent / "src" / "histoomnist").exists():
    ROOT = ROOT.parent

OUT = ROOT / "results/hest1k_human_visium_expression/evidence_package"
manifest = json.loads((OUT / "evidence_manifest.json").read_text(encoding="utf-8"))
benchmark = pd.read_csv(OUT / "benchmark_evidence_table.csv")
generalization = pd.read_csv(OUT / "generalization_status.csv")
biology = pd.read_csv(OUT / "biological_signature_evidence_table.csv")
claims = pd.read_csv(OUT / "claim_audit.csv")

assert manifest["status"] == "built"
assert manifest["rows"]["benchmark_table"] == len(benchmark)
assert manifest["rows"]["generalization_status"] == len(generalization)
assert manifest["rows"]["biological_signature_table"] == len(biology)
assert manifest["rows"]["claim_audit"] == len(claims)
manifest

{'status': 'built',
 'inputs': {'coverage95_diagnostics': 'results/hest1k_human_visium_expression/coverage95_diagnostics/run_summary.json',
  'histoomnist_benchmark': 'results/hest1k_human_visium_expression/benchmark_results/histoomnist_coverage95/summary.csv',
  'statistical_baselines': 'results/hest1k_human_visium_expression/statistical_baselines/summary.csv',
  'generalization_readiness': 'results/hest1k_human_visium_expression/generalization_readiness/run_summary.json',
  'generalization_smoke': 'results/hest1k_human_visium_expression/generalization_runs/smoke_sf_generated_tasks_epoch1_runner/summary.csv',
  'biological_signatures': 'results/hest1k_human_visium_expression/biological_signatures/run_summary.json'},
 'rows': {'benchmark_table': 10,
  'generalization_status': 13,
  'biological_signature_table': 10,
  'claim_audit': 6},
 'claim_status_counts': {'supported': 3,
  'supported_first_pass': 1,
  'prepared_not_final': 1,
  'not_supported_yet': 1},
 'outputs': {'benchmark_tabl

In [2]:
claim_status = claims["status"].value_counts().rename_axis("status").reset_index(name="n_claims")
assert "not_supported_yet" in set(claims["status"])
assert not claims[claims["status"].eq("not_supported_yet")]["limitation"].isna().any()
claim_status

,status,n_claims
0,supported,3
1,supported_first_pass,1
2,prepared_not_final,1
3,not_supported_yet,1


In [3]:
benchmark_view = benchmark[
    ["family", "method", "prediction_kind", "mean_gene_pearson", "valid_genes", "evidence_level", "caveat"]
].copy()
assert "smoke_only" in set(benchmark_view["evidence_level"])
assert benchmark_view.loc[benchmark_view["evidence_level"].eq("smoke_only"), "caveat"].str.contains("not report", case=False).all()
benchmark_view

,family,method,prediction_kind,mean_gene_pearson,valid_genes,evidence_level,caveat
0,HistoOmniST,histoomnist_rate,rate,1.583026e-01,16942,formal_internal,NaN
1,HistoOmniST,histoomnist_count_no_sf,count,3.277876e-01,16942,formal_internal,NaN
2,HistoOmniST,histoomnist_count_pred_sf,count,3.528980e-01,16942,formal_internal,NaN
3,HistoOmniST,histoomnist_count_oracle_sf,count,3.976097e-01,16942,formal_internal,NaN
4,Statistical/SF-only baseline,global_mean_rate,rate,4.988135e-11,2962,formal_internal_baseline,NaN
5,Statistical/SF-only baseline,global_sf_only_count_pred_sf,count,1.413055e-01,16942,formal_internal_baseline,NaN
6,Statistical/SF-only baseline,global_sf_only_count_oracle_sf,count,2.029177e-01,16942,formal_internal_baseline,NaN
7,Statistical/SF-only baseline,organ_sf_only_count_pred_sf,count,2.493534e-01,16942,formal_internal_baseline,NaN
8,Statistical/SF-only baseline,organ_sf_only_count_oracle_sf,count,3.135677e-01,16942,formal_internal_baseline,NaN
9,External baseline smoke,histogene_patch_h5,log1p_rate,5.165959e-05,16396,smoke_only,Do not report as formal external benchmark per...


In [4]:
generalization_view = generalization[["item", "status", "scope", "metric", "value", "caveat"]]
assert set(["formal_leave_organ_out_expression", "formal_leave_cohort_out_expression"]).issubset(set(generalization["item"]))
assert (generalization[generalization["item"].str.startswith("formal_leave_")]["status"] == "not_run").all()
generalization_view

,item,status,scope,metric,value,caveat
0,generalization_task_readiness,ready_for_selected_tasks,leave_slide_out|leave_organ_out|leave_cohort_out,NaN,NaN,Readiness only; no trained expression generali...
1,leave_cohort_out_ready_task_count,readiness_audit,leave_cohort_out,missing_asset_paths,0.000000,Counts reflect manifest/split/assets readiness...
2,leave_organ_out_ready_task_count,readiness_audit,leave_organ_out,missing_asset_paths,0.000000,Counts reflect manifest/split/assets readiness...
3,leave_slide_out_ready_task_count,readiness_audit,leave_slide_out,missing_asset_paths,0.000000,Counts reflect manifest/split/assets readiness...
4,sf_smoke_leave_slide_out_leave_slide_out,ok,leave_slide_out:leave_slide_out,log_sf_pearson,0.715960,One-epoch SF smoke; not formal generalization ...
5,sf_smoke_leave_organ_out_bowel,ok,leave_organ_out:Bowel,log_sf_pearson,0.386235,One-epoch SF smoke; not formal generalization ...
6,sf_smoke_leave_organ_out_brain,ok,leave_organ_out:Brain,log_sf_pearson,-0.020692,One-epoch SF smoke; not formal generalization ...
7,sf_smoke_leave_organ_out_breast,ok,leave_organ_out:Breast,log_sf_pearson,0.317577,One-epoch SF smoke; not formal generalization ...
8,sf_smoke_leave_cohort_out_colon_map_colon_mole...,ok,leave_cohort_out:COLON MAP: Colon Molecular At...,log_sf_pearson,0.188132,One-epoch SF smoke; not formal generalization ...
9,sf_smoke_leave_cohort_out_charting_the_heterog...,ok,leave_cohort_out:Charting the Heterogeneity of...,log_sf_pearson,0.224053,One-epoch SF smoke; not formal generalization ...


In [5]:
biology_view = biology[
    ["signature", "n_spots", "rate_signature_pearson", "count_pred_sf_pearson", "n_present_genes", "missing_genes"]
].sort_values("rate_signature_pearson", ascending=False)
assert len(biology_view) == 10
assert biology_view.iloc[0]["signature"] == "epithelial_tumor"
biology_view

,signature,n_spots,rate_signature_pearson,count_pred_sf_pearson,n_present_genes,missing_genes
0,epithelial_tumor,121421.0,0.866730,0.852484,6,KRT18|KRT19
1,stromal_ecm,117100.0,0.800195,0.771473,8,NaN
2,myeloid,117968.0,0.651851,0.627516,6,NaN
3,pan_immune,117968.0,0.613431,0.572889,8,NaN
4,b_cell,121421.0,0.586945,0.605659,5,NaN
5,proliferation,121421.0,0.580034,0.607876,5,NaN
6,endothelial,117968.0,0.498288,0.530502,6,NaN
7,hypoxia_stress,113647.0,0.496828,0.604934,5,VEGFA|LDHA
8,interferon_response,117968.0,0.490057,0.567955,7,NaN
9,t_cell,117968.0,0.442417,0.406769,7,NaN


In [6]:
report_path = OUT / "histo_omnist_coverage95_evidence_report.md"
report_text = report_path.read_text(encoding="utf-8")
assert "not formal generalization performance" in report_text
assert "Do not report as formal external benchmark performance" in report_text
assert "Next Required Evidence" in report_text
{"report_path": str(report_path.relative_to(ROOT)).replace("\\\\", "/"), "n_lines": len(report_text.splitlines())}

{'report_path': 'results\\hest1k_human_visium_expression\\evidence_package\\histo_omnist_coverage95_evidence_report.md',
 'n_lines': 72}